In [33]:
import numpy as np
import pandas as pd

In [34]:
from sklearn.model_selection import train_test_split,cross_val_score
from sklearn.tree import DecisionTreeClassifier

from sklearn.metrics import accuracy_score

from sklearn.compose import ColumnTransformer

In [35]:
df = pd.read_csv('used_car_listings.csv',usecols=[9,10,11])

In [36]:
df.head()

,mileage,price,condition
0,46134,19919.0,good
1,16109,19480.0,good
2,173239,4556.0,good
3,36810,11536.0,fair
4,87749,14098.0,good


In [39]:
df.isnull().sum()

mileage      0
price        0
condition    0
dtype: int64

In [38]:
df['condition'].fillna(df['condition'].value_counts().index[0],inplace=True)

C:\Users\Saqib Ali\AppData\Local\Temp\ipykernel_6712\2052693466.py:1: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df['condition'].fillna(df['condition'].value_counts().index[0],inplace=True)


In [40]:
x = df.iloc[:,0:2]
y = df.iloc[:,-1]

x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42)
x_train.head()

,mileage,price
932,108268,5430.0
1523,274780,1208.0
1111,0,28595.0
1679,95435,7609.0
123,77534,3971.0


In [41]:
# label Encoder
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()

y_train_transformed = le.fit_transform(y_train)
y_test_transformed = le.fit_transform(y_test)

y_transformed = le.fit_transform(y)

y_train_transformed = pd.DataFrame(y_train_transformed)
y_train_transformed.value_counts()

0
2    684
0    476
1    301
3     82
5     77
4     34
Name: count, dtype: int64

In [42]:
# Without Binarization

In [43]:
clf = DecisionTreeClassifier()

clf.fit(x_train,y_train_transformed)

y_pred = clf.predict(x_test)

accuracy_score(y_test_transformed,y_pred)

0.2584541062801932

In [44]:
np.mean(cross_val_score(clf,x,y_transformed,scoring='accuracy'))

0.2954755471336164

In [45]:
# Applying Binarization
from sklearn.preprocessing import Binarizer

trf = ColumnTransformer([
    ('bin',Binarizer(copy=False,threshold=1000),[0])
],remainder='passthrough')

x_train_trf = trf.fit_transform(x_train)
x_test_trf = trf.transform(x_test)

pd.DataFrame(x_train_trf, columns=['mileage','price']).sample(4)

,mileage,price
1525,1.0,4119.0
266,1.0,1948.0
928,1.0,13768.0
559,1.0,2766.0


In [46]:
# With binarization

In [47]:
clf = DecisionTreeClassifier()

clf.fit(x_train_trf,y_train_transformed)

y_pred2 = clf.predict(x_test_trf)

accuracy_score(y_test_transformed,y_pred2)

0.2898550724637681

In [48]:
x_transformed = trf.transform(x)
np.mean(cross_val_score(clf,x_transformed,y_transformed,scoring='accuracy'))

0.30077552023019966